# The borrowed inference strategies, each scored by the harness


**What this is.** Six inference techniques borrowed from large-scale profile inference (instruction 77), each written as a model the validation harness can score, and each scored on the two registry questions that have enough known answers for a frozen test.

**The rule.** A strategy ships when its frozen-test result beats the best baseline; it may fill a cell only when its frozen-test lower bound clears 0.90. A cross-validated number alone never fills anything. A strategy that could not be tested is listed with the reason.

**Why each has its own partition.** The harness lets one strategy definition look at a frozen partition once; a second, different strategy on the same partition would turn it into a tuning set and is refused. So every strategy below gets a fresh seed. The comparison between strategies therefore carries the noise of different partitions, which the cross-validated column (many folds) shows more steadily.

Built 2026-11-06.

In [1]:
import json, os, time
from pathlib import Path
import numpy as np, pandas as pd
from parascape import harness as H, borrowed_models as B, question_registry as Q
from parascape import datasets as ds, organisms as O, information_categories as IC
nodes = pd.read_parquet(os.path.join(REPO, 'parascape/data/nodes.parquet')).set_index('gene_id')
imap = pd.read_parquet(os.path.join(REPO, 'results/information_map_v0_2026_11_04/values.parquet'))
def family_of(column):
    found = ds.provenance(column, O.TOXOPLASMA)
    return found.key if found is not None else 'unregistered'
numeric = [c for c in nodes.columns if pd.api.types.is_numeric_dtype(nodes[c]) or pd.api.types.is_bool_dtype(nodes[c])]
families = pd.Series({c: family_of(c) for c in numeric}, name='family')
categories = {c: IC.categories_of(c) for c in numeric}
B.set_families(families, categories)
print(len(numeric), 'numeric columns in', families.nunique(), 'families')

411 numeric columns in 90 families


## 1 · The two questions with a frozen test

* `required_without_glucose`: measured for 7,393 genes by the glucose-withdrawal screen (`fit_no_glucose <= -2`); every pooled fitness screen is held out as evidence.

* `host_facing`: does the protein reach beyond the parasite plasma membrane into the host cell -- 585 genes with a curated answer in the information map (65 yes). Every place column and every proximity screen is held out.

In [2]:
QUESTIONS = {}
q = Q.get('required_without_glucose')
y = (nodes['fit_no_glucose'] <= -2.0)[nodes['fit_no_glucose'].notna()]
QUESTIONS[q.id] = (q, y)
q = Q.get('host_facing')
hf = imap['host_facing'].dropna().astype(bool)
QUESTIONS[q.id] = (q, hf[hf.index.isin(nodes.index)])
INPUTS = {}
for qid, (question, labels) in QUESTIONS.items():
    full = nodes.loc[labels.index, numeric].astype('float64')
    features = H.allowed_features(question, full, families)
    groups = nodes['orthogroup'].reindex(labels.index)
    groups = groups.where(groups.notna(), pd.Series({g: 'self:' + g for g in labels.index}))
    if DRYRUN:
        labels = pd.Series(np.random.default_rng(0).permutation(labels.to_numpy()), index=labels.index)
    INPUTS[qid] = (question, labels, features, groups)
    print(f'{qid}: {len(labels)} genes, {int(labels.sum())} yes; {features.shape[1]} allowed columns '
          f'in {families.reindex(features.columns).nunique()} families; frozen test possible: '
          f'{question.frozen_test_possible(len(labels))}')

required_without_glucose: 7393 genes, 1765 yes; 366 allowed columns in 74 families; frozen test possible: True
host_facing: 585 genes, 65 yes; 369 allowed columns in 84 families; frozen test possible: True


## 2 · The strategies

Boosted trees are the reference. The six borrowed strategies are `parascape.borrowed_models`; the docstring of each says what it does. Selective prediction is not a separate model: it is how the harness grades every one of them (coverage at the 0.90 bar).

In [3]:
STRATEGIES = [H.Strategy.of('boosted trees (reference)', 'hist_gbt', max_iter=150),
              H.Strategy.of('positive-unlabelled bagging', 'pu_bagging', n_bags=10),
              H.Strategy.of('inverse-propensity weighting', 'propensity_weighted'),
              H.Strategy.of('low-rank completion', 'low_rank_completion', rank=30),
              H.Strategy.of('per-family stacking', 'family_stacking'),
              H.Strategy.of('lookalike ranking', 'lookalike', k=10),
              H.Strategy.of('network propagation', 'graph_propagation', k=15)]
SEED0 = 10      # seeds 0-2 of required_without_glucose are spent; one fresh seed per strategy
pd.DataFrame([dict(strategy=s.name, model=s.model, seed=SEED0 + i, hash=s.identity[:12]) for i, s in enumerate(STRATEGIES)])

,strategy,model,seed,hash
0,boosted trees (reference),hist_gbt,10,1b4ace1f7077
1,positive-unlabelled bagging,pu_bagging,11,ae84888ce65d
2,inverse-propensity weighting,propensity_weighted,12,1911e96e9729
3,low-rank completion,low_rank_completion,13,2ad3985f1587
4,per-family stacking,family_stacking,14,73d50c75f685
5,lookalike ranking,lookalike,15,2748abd32da0
6,network propagation,graph_propagation,16,1a7566d721dd


## 3 · The runs

One `run_question` per strategy and question: nested 3 x 3 cross-validation inside train + tune, refit, isotonic calibration on the calibration partition, then the frozen partition once. Every use is appended to the holdout log.

In [4]:
rows, importance = [], []
for qid, (question, labels, features, groups) in INPUTS.items():
    for i, strategy in enumerate(STRATEGIES):
        t0 = time.time()
        r = H.run_question(question, labels=labels, features=features, families=families,
                           groups=groups, strategies=[strategy], seed=SEED0 + i,
                           outer_folds=3, inner_folds=3, importance_repeats=2,
                           log_path=LOG or os.path.join(REPO, H.HOLDOUT_LOG))
        cv, ft = r.cross_validated, r.frozen_test or {}
        base = r.baselines[r.baselines.partition == 'frozen_test'] if len(r.baselines) else r.baselines
        best = base.sort_values('precision_lower90', ascending=False).iloc[0] if len(base) else None
        rows.append(dict(question=qid, strategy=strategy.name, seed=SEED0 + i,
            cv_auc=round(cv.get('roc_auc', np.nan), 3), cv_ap=round(cv.get('average_precision', np.nan), 3),
            cv_coverage_at_bar=round(cv.get('coverage_at_bar', np.nan), 4),
            test_n=ft.get('n'), test_auc=round(ft.get('roc_auc', np.nan), 3),
            test_ap=round(ft.get('average_precision', np.nan), 3),
            test_calls=ft.get('calls'), test_precision=ft.get('precision'),
            test_lower90=round(ft.get('precision_lower90', np.nan), 3) if ft else np.nan,
            test_coverage_at_bar=round(ft.get('coverage_at_bar', np.nan), 4) if ft else np.nan,
            best_baseline=None if best is None else best['baseline'] if 'baseline' in best else best.iloc[0],
            best_baseline_lower90=np.nan if best is None else round(float(best['precision_lower90']), 3),
            calibration_error=round(r.calibration.get('calibration_error', np.nan), 3),
            may_fill=H.may_fill(r), flags=';'.join(r.flags), minutes=round((time.time() - t0) / 60, 1)))
        if r.importance is not None and len(r.importance):
            importance.append(r.importance.head(5).assign(question=qid, strategy=strategy.name))
        print(qid, '|', strategy.name, '|', rows[-1]['cv_auc'], rows[-1]['test_auc'], rows[-1]['minutes'], 'min')
scores = pd.DataFrame(rows)
scores.to_csv(os.path.join(OUT, 'scores.tsv'), sep='\t', index=False)
pd.concat(importance).to_csv(os.path.join(OUT, 'importance_top5.tsv'), sep='\t', index=False)
scores

required_without_glucose | boosted trees (reference) | 0.831 0.824 0.5 min
required_without_glucose | positive-unlabelled bagging | 0.833 0.808 2.4 min
required_without_glucose | inverse-propensity weighting | 0.81 0.836 4.2 min
required_without_glucose | low-rank completion | 0.812 0.813 0.6 min
required_without_glucose | per-family stacking | 0.817 0.824 10.5 min
required_without_glucose | lookalike ranking | 0.773 0.772 1.0 min
required_without_glucose | network propagation | 0.781 0.781 0.6 min
host_facing | boosted trees (reference) | 0.949 0.996 0.1 min
host_facing | positive-unlabelled bagging | 0.954 0.937 0.3 min
host_facing | inverse-propensity weighting | 0.946 0.967 0.8 min
host_facing | low-rank completion | 0.925 0.997 0.1 min
host_facing | per-family stacking | 0.937 0.977 5.9 min
host_facing | lookalike ranking | 0.952 0.995 0.1 min
host_facing | network propagation | 0.824 0.896 0.1 min


,question,strategy,seed,cv_auc,cv_ap,cv_coverage_at_bar,test_n,test_auc,test_ap,test_calls,test_precision,test_lower90,test_coverage_at_bar,best_baseline,best_baseline_lower90,calibration_error,may_fill,flags,minutes
0,required_without_glucose,boosted trees (reference),10,0.831,0.581,0.0098,1091,0.824,0.538,8,0.75,0.462,0.0073,prevalence-weighted random,0.2,0.029,False,,0.5
1,required_without_glucose,positive-unlabelled bagging,11,0.833,0.572,0.0132,1084,0.808,0.531,17,0.7647,0.584,0.0157,prevalence-weighted random,0.169,0.039,False,,2.4
2,required_without_glucose,inverse-propensity weighting,12,0.81,0.527,0.0351,1125,0.836,0.587,52,0.6923,0.596,0.0462,prevalence-weighted random,0.235,0.042,False,,4.2
3,required_without_glucose,low-rank completion,13,0.812,0.527,0.0408,1098,0.813,0.542,69,0.6667,0.584,0.0628,prevalence-weighted random,0.256,0.023,False,,0.6
4,required_without_glucose,per-family stacking,14,0.817,0.542,0,1087,0.824,0.578,0,NaN,NaN,0,prevalence-weighted random,0.199,0.034,False,,10.5
5,required_without_glucose,lookalike ranking,15,0.773,0.479,0.0418,1093,0.772,0.455,49,0.551,0.45,0.0448,prevalence-weighted random,0.222,0.053,False,,1
6,required_without_glucose,network propagation,16,0.781,0.487,0.0086,1115,0.781,0.451,10,0.7,0.448,0.009,prevalence-weighted random,0.179,0.04,False,,0.6
7,host_facing,boosted trees (reference),10,0.949,0.721,0.0803,87,0.996,0.962,5,1,0.631,0.0575,best single feature (fit_density_high),0.354,0.024,False,,0.1
8,host_facing,positive-unlabelled bagging,11,0.954,0.735,0.049,88,0.937,0.592,14,0.5,0.305,0.1591,prevalence-weighted random,0.061,0.09,False,,0.3
9,host_facing,inverse-propensity weighting,12,0.946,0.687,0.0706,87,0.967,0.624,3,0.6667,0.196,0.0345,best single feature (n_bcell_epitopes),0.1,0.024,False,,0.8


## 4 · Which ship

A strategy **ships** on a question when its frozen-test average precision beats the best baseline there and is not worse than the reference by more than 0.02 on the frozen test **and** on the cross-validated estimate. It **may fill** only when the frozen-test lower bound clears 0.90 -- `may_fill` above. Anything else is listed, with its numbers, as not better than what is already there.

In [5]:
ref = scores[scores.strategy == 'boosted trees (reference)'].set_index('question')
def verdict(row):
    if not np.isfinite(row.test_ap):
        return 'untested: no frozen test'
    beats_baseline = row.test_lower90 > row.best_baseline_lower90 if np.isfinite(row.best_baseline_lower90) else True
    r = ref.loc[row.question]
    close = row.test_ap >= r.test_ap - 0.02 and row.cv_ap >= r.cv_ap - 0.02
    if row.strategy.startswith('boosted'):
        return 'reference'
    return 'ships' if beats_baseline and close else ('beats baseline, worse than reference' if beats_baseline else 'does not beat the baseline')
scores['verdict'] = scores.apply(verdict, axis=1)
untested = pd.DataFrame([dict(strategy=k, verdict='untested', reason=v) for k, v in B.UNTESTED.items()])
scores.to_csv(os.path.join(OUT, 'scores.tsv'), sep='\t', index=False)
untested.to_csv(os.path.join(OUT, 'untested.tsv'), sep='\t', index=False)
scores[['question', 'strategy', 'cv_ap', 'test_ap', 'test_lower90', 'best_baseline_lower90', 'may_fill', 'verdict']]

,question,strategy,cv_ap,test_ap,test_lower90,best_baseline_lower90,may_fill,verdict
0,required_without_glucose,boosted trees (reference),0.581,0.538,0.462,0.2,False,reference
1,required_without_glucose,positive-unlabelled bagging,0.572,0.531,0.584,0.169,False,ships
2,required_without_glucose,inverse-propensity weighting,0.527,0.587,0.596,0.235,False,"beats baseline, worse than reference"
3,required_without_glucose,low-rank completion,0.527,0.542,0.584,0.256,False,"beats baseline, worse than reference"
4,required_without_glucose,per-family stacking,0.542,0.578,NaN,0.199,False,does not beat the baseline
5,required_without_glucose,lookalike ranking,0.479,0.455,0.45,0.222,False,"beats baseline, worse than reference"
6,required_without_glucose,network propagation,0.487,0.451,0.448,0.179,False,"beats baseline, worse than reference"
7,host_facing,boosted trees (reference),0.721,0.962,0.631,0.354,False,reference
8,host_facing,positive-unlabelled bagging,0.735,0.592,0.305,0.061,False,"beats baseline, worse than reference"
9,host_facing,inverse-propensity weighting,0.687,0.624,0.196,0.1,False,"beats baseline, worse than reference"


In [6]:
untested

,strategy,verdict,reason
0,selective_prediction,untested,"Built into the harness rather than a model: every strategy is graded by coverage at fixed precision (`coverage_at_bar`), with the fill threshold as the reject option."
1,multi_task,untested,"Needs auxiliary labels that may be used beside the target. The harness excludes every column of the target's own information category, and an auxiliary label of that category would bring it back in by another door. No pair of registry questions has yet been checked for which labels may stand beside which, so it is untested until the registry records that."
2,empirical_bayes_shrinkage,untested,"Needs a per-cell standard error. The only columns that carry one are the expression fold changes, which are already shrunk when they are built (`expression_traits`); the node table columns carry no uncertainty to shrink by, and rescaling a column changes nothing for a tree or a standardised linear model."
